# Mutation ↔ Chemical Relation-Wise Merge

> **Note:** The original notebook was an unfinished template — every data source was
> commented out and the merge referenced undefined `*_GENE_GENE` variables (it would have
> raised `NameError`), and its output path pointed at the wrong `GENE_GENE` directory.
> This is a clean scaffold with the correct schema, output path, and merge pipeline.
> **Add the actual source loads in section 2 before running.**

## 0. Configuration

In [15]:
import os
import pandas as pd

BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR  = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/MUTATION_CHEMICALENTITY/ALL_MUTATION_CHEMICALENTITY.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Lookup Dictionaries


In [ ]:
Pubchem = pd.read_pickle(DB_DIR + 'PUBCHEM/combined_df.pkl')
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
del Pubchem

## 2. Load KG Sources

**TODO:** add the real Mutation–Chemical source files. Example pattern below.

In [5]:
hald = pd.read_csv(PROC_DIR + 'hald/Mutation_Chemical_HALD.csv')
hald.columns = hald.columns.str.lower()
print(f"hald: {len(hald):,} rows | columns: {list(hald.columns)}")

hald['head_detail_name'] = hald['head']
hald['head_id_is'] = ''
hald['kg_type'] = 'Aging'
hald['species'] = 'Homo species'
hald.head(2)

hald: 27 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'tail_smiles_name', 'tail_detail_name', 'tail_id_is']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,tail_smiles_name,tail_detail_name,tail_id_is,head_detail_name,head_id_is,kg_type,species
0,rs2805533,Mutation_ChemicalEntity,Triglycerides,Mutation,associated,ChemicalEntity,HALD_KG,Triglycerides,Triglycerides,NaN,rs2805533,,Aging,Homo species
1,rs1260326,Mutation_ChemicalEntity,5793,Mutation,associated,ChemicalEntity,HALD_KG,C([C@@H]1[C@H]([C@@H]([C@H](C(O1)O)O)O)O)O,"(3R,4S,5S,6R)-6-(hydroxymethyl)oxane-2,3,4,5-t...",Pubchem,rs1260326,,Aging,Homo species


## 3. Check and Fix Duplicate Columns

In [6]:
SOURCE_DFS = [
    ('hald', hald),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[hald] ✓ no duplicates


In [8]:

for name, df in SOURCE_DFS:
    remaining = df.columns[df.columns.duplicated()].tolist()
    print(f"[{name}] {'✓ clean' if not remaining else '✗ dupes: ' + str(remaining)}")

[hald] ✓ clean


## 4. Align Schemas and Concatenate

In [9]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 27 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,rs2805533,Mutation_ChemicalEntity,Triglycerides,Mutation,associated,ChemicalEntity,HALD_KG,,NaN,rs2805533,Triglycerides,Aging,Homo species
1,rs1260326,Mutation_ChemicalEntity,5793,Mutation,associated,ChemicalEntity,HALD_KG,,Pubchem,rs1260326,"(3R,4S,5S,6R)-6-(hydroxymethyl)oxane-2,3,4,5-t...",Aging,Homo species


## 5. Standardise Fixed-Value Columns

In [10]:
final_df['head_type'] = 'Mutation'
final_df['tail_type'] = 'ChemicalEntity'
final_df['relation']  = 'Mutation_ChemicalEntity'
final_df['head_id_is'] = 'MutationalVariant'
# final_df['tail_id_is'] = 'Pubchem'   # set based on chemical ID scheme

print("Unique relation:",  set(final_df['relation']))
print("Unique kg_source:", set(final_df['kg_source']))

Unique relation: {'Mutation_ChemicalEntity'}
Unique kg_source: {'HALD_KG'}


## 6. Deduplicate and Add Schema Columns

In [11]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

final_df_dedup = final_df_dedup[REQUIRED_COLS]
final_df_dedup.head()

After deduplication: 25 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,rs10515522,Mutation_ChemicalEntity,5997,Mutation,associated,ChemicalEntity,HALD_KG,MutationalVariant,Pubchem,rs10515522,"(3S,8S,9S,10R,13R,14S,17R)-10,13-dimethyl-17-[...",Aging,Homo species
1,rs10801555,Mutation_ChemicalEntity,5997,Mutation,switch,ChemicalEntity,HALD_KG,MutationalVariant,Pubchem,rs10801555,"(3S,8S,9S,10R,13R,14S,17R)-10,13-dimethyl-17-[...",Aging,Homo species
2,rs10830963,Mutation_ChemicalEntity,5793,Mutation,associated,ChemicalEntity,HALD_KG,MutationalVariant,Pubchem,rs10830963,"(3R,4S,5S,6R)-6-(hydroxymethyl)oxane-2,3,4,5-t...",Aging,Homo species
3,rs11591147,Mutation_ChemicalEntity,5997,Mutation,associated,ChemicalEntity,HALD_KG,MutationalVariant,Pubchem,rs11591147,"(3S,8S,9S,10R,13R,14S,17R)-10,13-dimethyl-17-[...",Aging,Homo species
4,rs12126655,Mutation_ChemicalEntity,DB15263,Mutation,associated,ChemicalEntity,HALD_KG,MutationalVariant,Drugbank,rs12126655,Thyrotropin,Aging,Homo species


## 7. QC — NaN Check

In [12]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,8,0,8
head_detail_name,0,0,0


## 8. Save Output

In [16]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 25 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/MUTATION_CHEMICALENTITY/ALL_MUTATION_CHEMICALENTITY.csv
